In [ ]:
!pip -q install pyomo highspy

In [ ]:
import json
from collections import defaultdict

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pyomo.environ as pyo
from pyomo.opt import SolverFactory, TerminationCondition

from IPython.display import display

In [ ]:
# ============================================================
# Load the instance directly from GitHub
# ============================================================

JSON_URL = (
    "https://raw.githubusercontent.com/ceche1212/los_movimientos/refs/heads/main/data/los_movimientos_toy_instance.json"
)

response = requests.get(JSON_URL, timeout=30)
response.raise_for_status()

instance = response.json()

print("Instance loaded successfully")
print(f"Name: {instance['instance_name']}")
print(f"Vehicles: {len(instance['vehicles'])}")
print(f"Requests: {len(instance['requests'])}")
print(f"Service nodes: {2 * len(instance['requests'])}")

In [ ]:
# ============================================================
# Helper function for displaying time
# ============================================================

def minutes_to_clock(minutes):
    """
    Convert minutes after midnight into HH:MM format.
    """
    minutes = int(round(minutes))

    hours = minutes // 60
    remaining_minutes = minutes % 60

    return f"{hours:02d}:{remaining_minutes:02d}"


# ============================================================
# Create a readable request table
# ============================================================

request_rows = []

for request in instance["requests"]:

    pickup = request["pickup"]
    delivery = request["delivery"]

    request_rows.append({
        "request": request["id"],
        "movement type": request["movement_type"],
        "origin": pickup["location"],
        "destination": delivery["location"],
        "passengers": request["passengers"],
        "pickup window": (
            f"{minutes_to_clock(pickup['time_window'][0])}"
            f"–{minutes_to_clock(pickup['time_window'][1])}"
        ),
        "delivery window": (
            f"{minutes_to_clock(delivery['time_window'][0])}"
            f"–{minutes_to_clock(delivery['time_window'][1])}"
        )
    })


request_table = (
    pd.DataFrame(request_rows)
    .sort_values("pickup window")
    .reset_index(drop=True)
)

display(request_table)

In [ ]:
# ============================================================
# Basic sets
# ============================================================

vehicles = [
    vehicle["id"]
    for vehicle in instance["vehicles"]
]

request_ids = [
    request["id"]
    for request in instance["requests"]
]


# Fast access to the original JSON records
vehicle_record = {
    vehicle["id"]: vehicle
    for vehicle in instance["vehicles"]
}

request_record = {
    request["id"]: request
    for request in instance["requests"]
}


# ============================================================
# Pickup and delivery nodes
# ============================================================

pickup_node = {
    request_id: request_record[request_id]["pickup"]["node_id"]
    for request_id in request_ids
}

delivery_node = {
    request_id: request_record[request_id]["delivery"]["node_id"]
    for request_id in request_ids
}

passengers = {
    request_id: request_record[request_id]["passengers"]
    for request_id in request_ids
}


# ============================================================
# Vehicle-request compatibility
# ============================================================
#
# The current instance assumes that all pickup trucks can
# serve all requests.
#
# The code also supports a future JSON field called
# "compatible_vehicles".

compatible_vehicles = {
    request_id: request_record[request_id].get(
        "compatible_vehicles",
        vehicles.copy()
    )
    for request_id in request_ids
}


# ============================================================
# Service-node data
# ============================================================

service_nodes = []

node_location = {}
earliest = {}
latest = {}
service_time = {}
load_change = {}
node_type = {}
node_request = {}


for request_id in request_ids:

    request = request_record[request_id]

    # Pickup node
    pickup = request["pickup"]
    pickup_id = pickup["node_id"]

    service_nodes.append(pickup_id)

    node_location[pickup_id] = pickup["location"]
    earliest[pickup_id] = pickup["time_window"][0]
    latest[pickup_id] = pickup["time_window"][1]
    service_time[pickup_id] = pickup["service_time"]

    # Picking up personnel increases the vehicle load
    load_change[pickup_id] = request["passengers"]

    node_type[pickup_id] = "pickup"
    node_request[pickup_id] = request_id


    # Delivery node
    delivery = request["delivery"]
    delivery_id = delivery["node_id"]

    service_nodes.append(delivery_id)

    node_location[delivery_id] = delivery["location"]
    earliest[delivery_id] = delivery["time_window"][0]
    latest[delivery_id] = delivery["time_window"][1]
    service_time[delivery_id] = delivery["service_time"]

    # Dropping off personnel decreases the vehicle load
    load_change[delivery_id] = -request["passengers"]

    node_type[delivery_id] = "delivery"
    node_request[delivery_id] = request_id


# ============================================================
# Vehicle terminals and capacities
# ============================================================

start_node = {
    vehicle_id: f"START_{vehicle_id}"
    for vehicle_id in vehicles
}

end_node = {
    vehicle_id: f"END_{vehicle_id}"
    for vehicle_id in vehicles
}

capacity = {}
start_time = {}
end_time = {}

vehicle_nodes = {}


for vehicle_id in vehicles:

    vehicle = vehicle_record[vehicle_id]

    capacity[vehicle_id] = vehicle["capacity_passengers"]
    start_time[vehicle_id] = vehicle["start_time"]
    end_time[vehicle_id] = vehicle["end_time"]

    start = start_node[vehicle_id]
    end = end_node[vehicle_id]


    # Start terminal
    node_location[start] = vehicle["start_location"]

    # The vehicle begins exactly at its stated start time
    earliest[start] = vehicle["start_time"]
    latest[start] = vehicle["start_time"]

    service_time[start] = 0
    load_change[start] = 0

    node_type[start] = "start"
    node_request[start] = None


    # End terminal
    node_location[end] = vehicle["end_location"]

    earliest[end] = vehicle["start_time"]
    latest[end] = vehicle["end_time"]

    service_time[end] = 0
    load_change[end] = 0

    node_type[end] = "end"
    node_request[end] = None


    # Service nodes available to this vehicle
    compatible_service_nodes = [
        node
        for request_id in request_ids
        if vehicle_id in compatible_vehicles[request_id]
        for node in (
            pickup_node[request_id],
            delivery_node[request_id]
        )
    ]

    vehicle_nodes[vehicle_id] = (
        compatible_service_nodes
        + [start, end]
    )


print(f"Vehicles: {len(vehicles)}")
print(f"Requests: {len(request_ids)}")
print(f"Service nodes: {len(service_nodes)}")

In [ ]:
# ============================================================
# Distance and travel-time matrices
# ============================================================

travel_matrix = instance["travel_time_minutes"]
distance_matrix = instance["distance_km"]


# ============================================================
# Create feasible vehicle arcs
# ============================================================

arcs = []

arc_travel_time = {}
arc_distance = {}


for vehicle_id in vehicles:

    start = start_node[vehicle_id]
    end = end_node[vehicle_id]

    for origin_node in vehicle_nodes[vehicle_id]:

        # Nothing can leave the end terminal
        if origin_node == end:
            continue

        for destination_node in vehicle_nodes[vehicle_id]:

            # Nothing can enter the start terminal
            if destination_node == start:
                continue

            # Remove self-loops
            if origin_node == destination_node:
                continue


            origin_location = node_location[origin_node]
            destination_location = node_location[destination_node]

            travel = travel_matrix[
                origin_location
            ][
                destination_location
            ]

            distance = distance_matrix[
                origin_location
            ][
                destination_location
            ]


            # ------------------------------------------------
            # Remove arcs that can never satisfy the
            # destination time window
            # ------------------------------------------------

            earliest_possible_arrival = (
                earliest[origin_node]
                + service_time[origin_node]
                + travel
            )

            if earliest_possible_arrival > latest[destination_node]:
                continue


            arc = (
                vehicle_id,
                origin_node,
                destination_node
            )

            arcs.append(arc)

            arc_travel_time[arc] = travel
            arc_distance[arc] = distance


# ============================================================
# Incoming and outgoing adjacency lists
# ============================================================

outgoing = defaultdict(list)
incoming = defaultdict(list)


for vehicle_id, origin_node, destination_node in arcs:

    outgoing[
        vehicle_id,
        origin_node
    ].append(destination_node)

    incoming[
        vehicle_id,
        destination_node
    ].append(origin_node)


# ============================================================
# Additional indexed sets
# ============================================================

vehicle_node_pairs = [
    (vehicle_id, node)
    for vehicle_id in vehicles
    for node in vehicle_nodes[vehicle_id]
]

compatible_pairs = [
    (vehicle_id, request_id)
    for request_id in request_ids
    for vehicle_id in compatible_vehicles[request_id]
]

service_node_pairs = [
    (vehicle_id, node)
    for vehicle_id in vehicles
    for node in service_nodes
    if node in vehicle_nodes[vehicle_id]
]


print(f"Number of feasible arcs: {len(arcs)}")

In [ ]:
# ============================================================
# Big-M values for time propagation
# ============================================================

M_time = {
    (vehicle_id, origin_node, destination_node): max(
        0,
        latest[origin_node]
        + service_time[origin_node]
        + arc_travel_time[
            vehicle_id,
            origin_node,
            destination_node
        ]
        - earliest[destination_node]
    )
    for vehicle_id, origin_node, destination_node in arcs
}


# ============================================================
# Big-M values for load propagation
# ============================================================

M_load = {
    (vehicle_id, origin_node, destination_node): (
        capacity[vehicle_id]
        + max(
            0,
            load_change[destination_node]
        )
    )
    for vehicle_id, origin_node, destination_node in arcs
}


# ============================================================
# Big-M values for pickup-before-delivery constraints
# ============================================================

M_precedence = {
    (vehicle_id, request_id): max(
        0,
        latest[pickup_node[request_id]]
        - earliest[delivery_node[request_id]]
    )
    for vehicle_id, request_id in compatible_pairs
}

In [ ]:
# ============================================================
# Model
# ============================================================

model = pyo.ConcreteModel(
    name="Los_Movimientos_PDPTW"
)


# ============================================================
# Sets
# ============================================================

model.K = pyo.Set(
    initialize=vehicles,
    ordered=True
)

model.R = pyo.Set(
    initialize=request_ids,
    ordered=True
)

model.N = pyo.Set(
    initialize=service_nodes,
    ordered=True
)

model.KV = pyo.Set(
    dimen=2,
    initialize=vehicle_node_pairs
)

model.KR = pyo.Set(
    dimen=2,
    initialize=compatible_pairs
)

model.KN = pyo.Set(
    dimen=2,
    initialize=service_node_pairs
)

model.A = pyo.Set(
    dimen=3,
    initialize=arcs
)


# ============================================================
# Parameters
# ============================================================

model.capacity = pyo.Param(
    model.K,
    initialize=capacity
)

model.earliest = pyo.Param(
    model.KV,
    initialize={
        (vehicle_id, node): earliest[node]
        for vehicle_id, node in vehicle_node_pairs
    }
)

model.latest = pyo.Param(
    model.KV,
    initialize={
        (vehicle_id, node): latest[node]
        for vehicle_id, node in vehicle_node_pairs
    }
)

model.service_time = pyo.Param(
    model.KV,
    initialize={
        (vehicle_id, node): service_time[node]
        for vehicle_id, node in vehicle_node_pairs
    }
)

model.load_change = pyo.Param(
    model.KV,
    initialize={
        (vehicle_id, node): load_change[node]
        for vehicle_id, node in vehicle_node_pairs
    }
)

model.travel_time = pyo.Param(
    model.A,
    initialize=arc_travel_time
)

model.distance = pyo.Param(
    model.A,
    initialize=arc_distance
)

model.M_time = pyo.Param(
    model.A,
    initialize=M_time
)

model.M_load = pyo.Param(
    model.A,
    initialize=M_load
)

model.M_precedence = pyo.Param(
    model.KR,
    initialize=M_precedence
)


# ============================================================
# Objective-function weights
# ============================================================

model.alpha = pyo.Param(
    initialize=1.0
)

model.beta = pyo.Param(
    initialize=0.05
)

model.gamma = pyo.Param(
    initialize=10_000.0
)

In [ ]:
# ============================================================
# Decision variables
# ============================================================

# 1 if vehicle k travels directly from node i to node j
model.x = pyo.Var(
    model.A,
    domain=pyo.Binary
)

# 1 if request r is left unserved
model.z = pyo.Var(
    model.R,
    domain=pyo.Binary
)

# Service-start time at each node
model.S = pyo.Var(
    model.KV,
    domain=pyo.NonNegativeReals
)

# Number of passengers onboard after servicing each node
model.L = pyo.Var(
    model.KV,
    domain=pyo.NonNegativeReals
)

In [ ]:
# ============================================================
# Constraint (1)
# Every request is either served or left unserved
# ============================================================

def request_assignment_rule(model, request_id):

    pickup = pickup_node[request_id]

    served = sum(
        model.x[
            vehicle_id,
            pickup,
            destination
        ]
        for vehicle_id in compatible_vehicles[request_id]
        for destination in outgoing[
            vehicle_id,
            pickup
        ]
    )

    return served + model.z[request_id] == 1


model.RequestAssignment = pyo.Constraint(
    model.R,
    rule=request_assignment_rule
)


# ============================================================
# Constraint (2)
# Pickup and delivery are performed by the same vehicle
# ============================================================

def same_vehicle_rule(
    model,
    vehicle_id,
    request_id
):

    pickup = pickup_node[request_id]
    delivery = delivery_node[request_id]

    pickup_visit = sum(
        model.x[
            vehicle_id,
            pickup,
            destination
        ]
        for destination in outgoing[
            vehicle_id,
            pickup
        ]
    )

    delivery_visit = sum(
        model.x[
            vehicle_id,
            origin,
            delivery
        ]
        for origin in incoming[
            vehicle_id,
            delivery
        ]
    )

    return pickup_visit == delivery_visit


model.SameVehicle = pyo.Constraint(
    model.KR,
    rule=same_vehicle_rule
)


# ============================================================
# Constraint (3)
# Every vehicle leaves its start terminal once
# ============================================================

def start_terminal_rule(
    model,
    vehicle_id
):

    start = start_node[vehicle_id]

    return sum(
        model.x[
            vehicle_id,
            start,
            destination
        ]
        for destination in outgoing[
            vehicle_id,
            start
        ]
    ) == 1


model.StartTerminal = pyo.Constraint(
    model.K,
    rule=start_terminal_rule
)


# ============================================================
# Constraint (4)
# Every vehicle enters its end terminal once
# ============================================================

def end_terminal_rule(
    model,
    vehicle_id
):

    end = end_node[vehicle_id]

    return sum(
        model.x[
            vehicle_id,
            origin,
            end
        ]
        for origin in incoming[
            vehicle_id,
            end
        ]
    ) == 1


model.EndTerminal = pyo.Constraint(
    model.K,
    rule=end_terminal_rule
)


# ============================================================
# Constraint (5)
# Flow conservation
# ============================================================

def flow_conservation_rule(
    model,
    vehicle_id,
    node
):

    incoming_flow = sum(
        model.x[
            vehicle_id,
            origin,
            node
        ]
        for origin in incoming[
            vehicle_id,
            node
        ]
    )

    outgoing_flow = sum(
        model.x[
            vehicle_id,
            node,
            destination
        ]
        for destination in outgoing[
            vehicle_id,
            node
        ]
    )

    return incoming_flow == outgoing_flow


model.FlowConservation = pyo.Constraint(
    model.KN,
    rule=flow_conservation_rule
)


# ============================================================
# Constraint (6)
# Time propagation along selected arcs
# ============================================================

def time_propagation_rule(
    model,
    vehicle_id,
    origin,
    destination
):

    return (
        model.S[vehicle_id, origin]
        + model.service_time[vehicle_id, origin]
        + model.travel_time[
            vehicle_id,
            origin,
            destination
        ]
        <=
        model.S[vehicle_id, destination]
        + model.M_time[
            vehicle_id,
            origin,
            destination
        ]
        * (
            1
            - model.x[
                vehicle_id,
                origin,
                destination
            ]
        )
    )


model.TimePropagation = pyo.Constraint(
    model.A,
    rule=time_propagation_rule
)


# ============================================================
# Constraint (7)
# Time windows
# ============================================================

def earliest_time_rule(
    model,
    vehicle_id,
    node
):

    return (
        model.S[vehicle_id, node]
        >= model.earliest[vehicle_id, node]
    )


def latest_time_rule(
    model,
    vehicle_id,
    node
):

    return (
        model.S[vehicle_id, node]
        <= model.latest[vehicle_id, node]
    )


model.EarliestTime = pyo.Constraint(
    model.KV,
    rule=earliest_time_rule
)

model.LatestTime = pyo.Constraint(
    model.KV,
    rule=latest_time_rule
)


# Vehicles begin operating at their stated start time
def vehicle_start_time_rule(
    model,
    vehicle_id
):

    return (
        model.S[
            vehicle_id,
            start_node[vehicle_id]
        ]
        == start_time[vehicle_id]
    )


model.VehicleStartTime = pyo.Constraint(
    model.K,
    rule=vehicle_start_time_rule
)


# ============================================================
# Constraint (8)
# Pickup before delivery
# ============================================================

def pickup_before_delivery_rule(
    model,
    vehicle_id,
    request_id
):

    pickup = pickup_node[request_id]
    delivery = delivery_node[request_id]

    served_by_vehicle = sum(
        model.x[
            vehicle_id,
            pickup,
            destination
        ]
        for destination in outgoing[
            vehicle_id,
            pickup
        ]
    )

    return (
        model.S[vehicle_id, pickup]
        <=
        model.S[vehicle_id, delivery]
        + model.M_precedence[
            vehicle_id,
            request_id
        ]
        * (
            1 - served_by_vehicle
        )
    )


model.PickupBeforeDelivery = pyo.Constraint(
    model.KR,
    rule=pickup_before_delivery_rule
)


# ============================================================
# Constraint (9)
# Passenger-load propagation
# ============================================================

def load_propagation_rule(
    model,
    vehicle_id,
    origin,
    destination
):

    return (
        model.L[vehicle_id, origin]
        + model.load_change[
            vehicle_id,
            destination
        ]
        <=
        model.L[vehicle_id, destination]
        + model.M_load[
            vehicle_id,
            origin,
            destination
        ]
        * (
            1
            - model.x[
                vehicle_id,
                origin,
                destination
            ]
        )
    )


model.LoadPropagation = pyo.Constraint(
    model.A,
    rule=load_propagation_rule
)


# ============================================================
# Constraint (10)
# Vehicle passenger capacity
# ============================================================

def vehicle_capacity_rule(
    model,
    vehicle_id,
    node
):

    return (
        model.L[vehicle_id, node]
        <= model.capacity[vehicle_id]
    )


model.VehicleCapacity = pyo.Constraint(
    model.KV,
    rule=vehicle_capacity_rule
)


# ============================================================
# Constraint (11)
# Vehicles start and finish empty
# ============================================================

def start_load_rule(
    model,
    vehicle_id
):

    return (
        model.L[
            vehicle_id,
            start_node[vehicle_id]
        ]
        == 0
    )


def end_load_rule(
    model,
    vehicle_id
):

    return (
        model.L[
            vehicle_id,
            end_node[vehicle_id]
        ]
        == 0
    )


model.StartLoad = pyo.Constraint(
    model.K,
    rule=start_load_rule
)

model.EndLoad = pyo.Constraint(
    model.K,
    rule=end_load_rule
)

In [ ]:
# ============================================================
# Objective components
# ============================================================

model.TotalDistance = pyo.Expression(
    expr=sum(
        model.distance[
            vehicle_id,
            origin,
            destination
        ]
        * model.x[
            vehicle_id,
            origin,
            destination
        ]
        for vehicle_id, origin, destination in model.A
    )
)


model.TotalDuration = pyo.Expression(
    expr=sum(
        model.S[
            vehicle_id,
            end_node[vehicle_id]
        ]
        - start_time[vehicle_id]
        for vehicle_id in model.K
    )
)


model.UnservedCount = pyo.Expression(
    expr=sum(
        model.z[request_id]
        for request_id in model.R
    )
)


# ============================================================
# Objective function
# ============================================================

model.TotalCost = pyo.Objective(
    expr=(
        model.alpha
        * model.TotalDistance

        + model.beta
        * model.TotalDuration

        + model.gamma
        * model.UnservedCount
    ),
    sense=pyo.minimize
)

In [ ]:
# ============================================================
# Solve with HiGHS
# ============================================================

solver = SolverFactory("appsi_highs")


# Solver parameters
solver.highs_options["output_flag"] = True
solver.highs_options["log_to_console"] = True

solver.highs_options["mip_rel_gap"] = 0.0
solver.highs_options["time_limit"] = 120

In [ ]:
results = solver.solve(
    model,
    tee=True
)

termination_condition = (
    results.solver.termination_condition
)

print(
    "\nTermination condition:",
    termination_condition
)


if termination_condition != TerminationCondition.optimal:

    raise RuntimeError(
        "HiGHS did not find an optimal solution. "
        f"Termination condition: {termination_condition}"
    )

In [ ]:
# ============================================================
# Objective-function results
# ============================================================

objective_value = pyo.value(
    model.TotalCost
)

total_distance = pyo.value(
    model.TotalDistance
)

total_duration = pyo.value(
    model.TotalDuration
)

unserved_count = int(
    round(
        pyo.value(
            model.UnservedCount
        )
    )
)


print("Optimal solution found")

print(
    f"\nObjective value: "
    f"{objective_value:.2f}"
)

print(
    f"Total distance: "
    f"{total_distance:.1f} km"
)

print(
    f"Total vehicle operating time: "
    f"{total_duration:.0f} minutes"
)

print(
    f"Unserved requests: "
    f"{unserved_count}"
)


unserved_requests = [
    request_id
    for request_id in model.R
    if pyo.value(
        model.z[request_id]
    ) > 0.5
]


if unserved_requests:

    print(
        "Requests left unserved:",
        unserved_requests
    )

else:

    print(
        "All transportation requests were served."
    )

In [ ]:
# ============================================================
# Identify the selected successor of each visited node
# ============================================================

selected_successor = {}


for vehicle_id, origin, destination in model.A:

    if pyo.value(
        model.x[
            vehicle_id,
            origin,
            destination
        ]
    ) > 0.5:

        selected_successor[
            vehicle_id,
            origin
        ] = destination


# ============================================================
# Reconstruct each complete route
# ============================================================

routes = {}


for vehicle_id in vehicles:

    route = [
        start_node[vehicle_id]
    ]

    current_node = start_node[vehicle_id]

    visited_nodes = {
        current_node
    }


    while current_node != end_node[vehicle_id]:

        if (
            vehicle_id,
            current_node
        ) not in selected_successor:

            raise RuntimeError(
                f"Route extraction failed for "
                f"{vehicle_id} at node {current_node}."
            )


        current_node = selected_successor[
            vehicle_id,
            current_node
        ]

        route.append(
            current_node
        )


        if (
            current_node in visited_nodes
            and current_node != end_node[vehicle_id]
        ):

            raise RuntimeError(
                f"A cycle was detected in the route "
                f"of {vehicle_id}."
            )


        visited_nodes.add(
            current_node
        )


        if len(route) > len(vehicle_nodes[vehicle_id]) + 1:

            raise RuntimeError(
                f"Route extraction exceeded the "
                f"expected size for {vehicle_id}."
            )


    routes[vehicle_id] = route

In [ ]:
# ============================================================
# Route summary
# ============================================================

summary_rows = []


for vehicle_id, route in routes.items():

    route_distance = sum(
        arc_distance[
            vehicle_id,
            origin,
            destination
        ]
        for origin, destination in zip(
            route[:-1],
            route[1:]
        )
    )


    finish_time = pyo.value(
        model.S[
            vehicle_id,
            end_node[vehicle_id]
        ]
    )


    requests_served = sum(
        node_type[node] == "pickup"
        for node in route
    )


    physical_route = " → ".join(
        node_location[node]
        for node in route
    )


    summary_rows.append({
        "vehicle": vehicle_id,
        "requests served": requests_served,
        "distance (km)": round(
            route_distance,
            1
        ),
        "finish time": minutes_to_clock(
            finish_time
        ),
        "route duration (min)": round(
            finish_time
            - start_time[vehicle_id],
            1
        ),
        "physical route": physical_route
    })


route_summary = pd.DataFrame(
    summary_rows
)

display(route_summary)

In [ ]:
route_summary['physical route'][0]

In [ ]:
# ============================================================
# Detailed vehicle schedule
# ============================================================

schedule_rows = []


for vehicle_id, route in routes.items():

    previous_node = None


    for sequence, node in enumerate(route):

        if previous_node is None:

            travel_from_previous = 0

        else:

            travel_from_previous = (
                arc_travel_time[
                    vehicle_id,
                    previous_node,
                    node
                ]
            )


        service_start = pyo.value(
            model.S[
                vehicle_id,
                node
            ]
        )

        passengers_onboard = pyo.value(
            model.L[
                vehicle_id,
                node
            ]
        )


        schedule_rows.append({
            "vehicle": vehicle_id,
            "stop": sequence,
            "node": node,
            "event": node_type[node],
            "request": node_request[node] or "",
            "location": node_location[node],
            "travel from previous (min)": (
                travel_from_previous
            ),
            "service start": minutes_to_clock(
                service_start
            ),
            "service time (min)": (
                service_time[node]
            ),
            "passenger change": (
                load_change[node]
            ),
            "passengers onboard": round(
                max(
                    0,
                    passengers_onboard
                ),
                6
            )
        })


        previous_node = node


schedule_table = (
    pd.DataFrame(schedule_rows)
    .sort_values(
        ["vehicle", "stop"]
    )
    .reset_index(drop=True)
)


display(schedule_table)

In [ ]:
# ============================================================
# Print the routes
# ============================================================

for vehicle_id, route in routes.items():

    print(
        "\n"
        + "=" * 70
    )

    print(
        f"Route for {vehicle_id}"
    )

    print(
        "=" * 70
    )


    for node in route:

        service_start = pyo.value(
            model.S[
                vehicle_id,
                node
            ]
        )

        onboard = pyo.value(
            model.L[
                vehicle_id,
                node
            ]
        )


        if node_request[node] is None:

            request_text = ""

        else:

            request_text = (
                f" | Request: {node_request[node]}"
            )


        print(
            f"{minutes_to_clock(service_start)}"
            f" | {node_location[node]:<18}"
            f" | {node_type[node]:<8}"
            f"{request_text}"
            f" | Passengers onboard: "
            f"{max(0, onboard):.0f}"
        )

In [ ]:
# ============================================================
# Plot each vehicle route on a separate axis
# ============================================================

coordinates = instance["locations"]

vehicle_ids = list(routes.keys())

if len(vehicle_ids) != 3:
    raise ValueError(
        "This visualization expects exactly three vehicles, "
        f"but {len(vehicle_ids)} were found."
    )


# Use common coordinate limits across all three plots
all_x = [
    location["x_km"]
    for location in coordinates.values()
]

all_y = [
    location["y_km"]
    for location in coordinates.values()
]

x_margin = 5
y_margin = 5


fig, axes = plt.subplots(
    1,
    3,
    figsize=(20, 7),
    sharex=True,
    sharey=True
)


for ax, vehicle_id in zip(axes, vehicle_ids):

    route = routes[vehicle_id]

    # --------------------------------------------------------
    # Plot all physical locations
    # --------------------------------------------------------

    for location_id, location in coordinates.items():

        ax.scatter(
            location["x_km"],
            location["y_km"],
            s=100,
            zorder=3
        )

        ax.text(
            location["x_km"] + 0.8,
            location["y_km"] + 0.8,
            location["name"],
            fontsize=9
        )


    # --------------------------------------------------------
    # Extract the coordinates of the vehicle route
    # --------------------------------------------------------

    x_values = [
        coordinates[
            node_location[node]
        ]["x_km"]
        for node in route
    ]

    y_values = [
        coordinates[
            node_location[node]
        ]["y_km"]
        for node in route
    ]


    # --------------------------------------------------------
    # Plot the route
    # --------------------------------------------------------

    ax.plot(
        x_values,
        y_values,
        marker="o",
        linewidth=2,
        zorder=2
    )


    # --------------------------------------------------------
    # Add arrows to indicate travel direction
    # --------------------------------------------------------

    for (
        x_origin,
        y_origin,
        x_destination,
        y_destination
    ) in zip(
        x_values[:-1],
        y_values[:-1],
        x_values[1:],
        y_values[1:]
    ):

        # Do not draw an arrow when two consecutive service
        # nodes correspond to the same physical location
        if (
            x_origin,
            y_origin
        ) == (
            x_destination,
            y_destination
        ):
            continue

        ax.annotate(
            "",
            xy=(
                x_destination,
                y_destination
            ),
            xytext=(
                x_origin,
                y_origin
            ),
            arrowprops={
                "arrowstyle": "->",
                "alpha": 0.5,
                "linewidth": 1.5
            }
        )


    # --------------------------------------------------------
    # Add the visit sequence beside each route stop
    # --------------------------------------------------------

    for sequence, node in enumerate(route):

        x = coordinates[
            node_location[node]
        ]["x_km"]

        y = coordinates[
            node_location[node]
        ]["y_km"]

        ax.annotate(
            str(sequence),
            xy=(x, y),
            xytext=(-8, -12),
            textcoords="offset points",
            fontsize=8,
            fontweight="bold"
        )


    # --------------------------------------------------------
    # Route statistics
    # --------------------------------------------------------

    route_distance = sum(
        arc_distance[
            vehicle_id,
            origin,
            destination
        ]
        for origin, destination in zip(
            route[:-1],
            route[1:]
        )
    )

    finish_time = pyo.value(
        model.S[
            vehicle_id,
            end_node[vehicle_id]
        ]
    )


    ax.set_title(
        f"{vehicle_id}\n"
        f"{route_distance:.1f} km | "
        f"Finish: {minutes_to_clock(finish_time)}"
    )

    ax.set_xlabel(
        "x coordinate (km)"
    )

    ax.grid(
        True,
        alpha=0.3
    )

    ax.set_xlim(
        min(all_x) - x_margin,
        max(all_x) + x_margin
    )

    ax.set_ylim(
        min(all_y) - y_margin,
        max(all_y) + y_margin
    )

    ax.set_aspect(
        "equal",
        adjustable="box"
    )


axes[0].set_ylabel(
    "y coordinate (km)"
)



plt.tight_layout()

plt.show()